In [1]:
!pip install pyabsa transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.2/574.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 15.7 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=b66211069574c83173a3bc2d3f00adc92312d84208f7bb6aaceae48656f39f7b
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
import pandas as pd

df = pd.read_csv('olist_processado.csv')
print("Shape:", df.shape)
print("\nColunas:", df.columns.tolist())
print("\nPrimeiras linhas:")
df.head()

Shape: (35246, 4)

Colunas: ['texto', 'sentimento', 'aspecto', 'sentimento_aspecto']

Primeiras linhas:


,texto,sentimento,aspecto,sentimento_aspecto
0,recebi bem antes do prazo estipulado,positivo,NaN,NaN
1,parabéns lojas lannister adorei comprar pela i...,positivo,NaN,NaN
2,aparelho eficiente no site a marca do aparelho...,positivo,NaN,NaN
3,mas um pouco travando pelo valor ta boa,positivo,NaN,NaN
4,vendedor confiável produto ok e entrega antes ...,positivo,NaN,NaN


In [3]:
# O PyABSA precisa do sentimento como número
# positivo = 1, neutro = 0, negativo = -1

def mapear_sentimento(s):
    if s == 'positivo':
        return 1
    elif s == 'neutro':
        return 0
    else:
        return -1

df['sentimento_id'] = df['sentimento'].apply(mapear_sentimento)

print("Distribuição:")
print(df['sentimento'].value_counts())

Distribuição:
sentimento
positivo    21457
negativo    10577
neutro       3212
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df[['texto', 'sentimento', 'sentimento_id']],
    test_size=0.2,
    random_state=42,
    stratify=df['sentimento']
)

print("Treino:", train_df.shape)
print("Teste:", test_df.shape)
print("\nDistribuição treino:")
print(train_df['sentimento'].value_counts())

Treino: (28196, 3)
Teste: (7050, 3)

Distribuição treino:
sentimento
positivo    17165
negativo     8461
neutro       2570
Name: count, dtype: int64


In [5]:
import os

os.makedirs('datasets/sentiment', exist_ok=True)

def salvar_pyabsa(df, arquivo):
    with open(arquivo, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            f.write(f"{row['texto']}${row['sentimento_id']}\n")

salvar_pyabsa(train_df, 'datasets/sentiment/train.txt')
salvar_pyabsa(test_df, 'datasets/sentiment/test.txt')

print("Arquivos salvos!")
print("Treino:", sum(1 for _ in open('datasets/sentiment/train.txt')), "linhas")
print("Teste:", sum(1 for _ in open('datasets/sentiment/test.txt')), "linhas")

Arquivos salvos!
Treino: 28196 linhas
Teste: 7050 linhas


In [6]:
from pyabsa import ModelSaveOption, DeviceTypeOption
from pyabsa.tasks.TextClassification import TCTrainer, TCConfigManager, TCDatasetList

config = TCConfigManager.get_tc_config_english()
config.model = 'BERT'
config.pretrained_bert = 'neuralmind/bert-base-portuguese-cased'
config.num_epoch = 3
config.batch_size = 16
config.max_seq_len = 128
config.seed = 42
config.log_step = 100
config.evaluate_begin = 0
config.save_mode = ModelSaveOption.SAVE_MODEL_STATE_DICT
config.device = DeviceTypeOption.AUTO

trainer = TCTrainer(
    config=config,
    dataset='datasets/sentiment',
    checkpoint_save_mode=ModelSaveOption.SAVE_MODEL_STATE_DICT,
    auto_device=True
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


No CUDA GPU found in your device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1480: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `t

[2026-05-17 19:58:34] (2.4.2) PyABSA(2.4.2): If your code crashes on Colab, please use the GPU runtime. Then run "pip install pyabsa[dev] -U" and restart the kernel.
Or if it does not work, you can use v1.x versions, e.g., pip install pyabsa<2.0 -U




Try to downgrade transformers<=4.29.0.




[2026-05-17 19:58:36] (2.4.2) Set Model Device: cpu
[2026-05-17 19:58:36] (2.4.2) Device Name: Unknown


AttributeError: 'str' object has no attribute '__name__'

In [7]:
from pyabsa import ModelSaveOption, DeviceTypeOption
from pyabsa.tasks.TextClassification import TCTrainer, TCConfigManager, BERTTCModelList

config = TCConfigManager.get_tc_config_english()
config.model = BERTTCModelList.BERT
config.pretrained_bert = 'neuralmind/bert-base-portuguese-cased'
config.num_epoch = 3
config.batch_size = 16
config.max_seq_len = 128
config.seed = 42
config.log_step = 100
config.evaluate_begin = 0
config.save_mode = ModelSaveOption.SAVE_MODEL_STATE_DICT

trainer = TCTrainer(
    config=config,
    dataset='datasets/sentiment',
    checkpoint_save_mode=ModelSaveOption.SAVE_MODEL_STATE_DICT,
    auto_device=True
)

[2026-05-17 19:59:58] (2.4.2) Set Model Device: cpu
[2026-05-17 19:59:58] (2.4.2) Device Name: Unknown


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=9908) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


2026-05-17 19:59:59,549 INFO: PyABSA version: 2.4.2


...

INFO:bert_mlp:Torch version: 2.10.0+cpu+cudaNone


2026-05-17 19:59:59,556 INFO: Device: Unknown


INFO:bert_mlp:Device: Unknown


[2026-05-17 19:59:59] (2.4.2) Try to load ['datasets/sentiment'] dataset from local disk


FileNotFoundError: [Errno 2] No such file or directory: ''

In [8]:
import os
print(os.listdir('/content'))

['.config', 'olist_processado.csv', 'logs', 'datasets', 'sample_data']


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Carregar dados
df = pd.read_csv('olist_processado.csv')

# Mapear sentimento
def mapear_sentimento(s):
    if s == 'positivo':
        return 1
    elif s == 'neutro':
        return 0
    else:
        return -1

df['sentimento_id'] = df['sentimento'].apply(mapear_sentimento)

# Dividir treino e teste
train_df, test_df = train_test_split(
    df[['texto', 'sentimento', 'sentimento_id']],
    test_size=0.2,
    random_state=42,
    stratify=df['sentimento']
)

# Salvar no formato PyABSA
os.makedirs('datasets/sentiment', exist_ok=True)

def salvar_pyabsa(df, arquivo):
    with open(arquivo, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            f.write(f"{row['texto']}${row['sentimento_id']}\n")

salvar_pyabsa(train_df, 'datasets/sentiment/train.txt')
salvar_pyabsa(test_df, 'datasets/sentiment/test.txt')

print("✅ Dados preparados!")
print("Treino:", len(train_df), "linhas")
print("Teste:", len(test_df), "linhas")

/usr/lib/python3.12/multiprocessing/pool.py:268: ResourceWarning: unclosed running multiprocessing pool <multiprocessing.pool.Pool state=RUN pool_size=1>
  _warn(f"unclosed running multiprocessing pool {self!r}",
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Dados preparados!
Treino: 28196 linhas
Teste: 7050 linhas


In [10]:
from pyabsa.tasks.TextClassification import TCTrainer, TCConfigManager, BERTTCModelList
from pyabsa import ModelSaveOption

config = TCConfigManager.get_tc_config_english()
config.model = BERTTCModelList.BERT
config.pretrained_bert = 'neuralmind/bert-base-portuguese-cased'
config.num_epoch = 3
config.batch_size = 16
config.max_seq_len = 128
config.seed = 42
config.log_step = 100
config.evaluate_begin = 0
config.save_mode = ModelSaveOption.SAVE_MODEL_STATE_DICT

trainer = TCTrainer(
    config=config,
    dataset='datasets/sentiment',
    checkpoint_save_mode=ModelSaveOption.SAVE_MODEL_STATE_DICT,
    auto_device=True
)

[2026-05-17 20:03:32] (2.4.2) Set Model Device: cpu
[2026-05-17 20:03:32] (2.4.2) Device Name: Unknown


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=9908) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


2026-05-17 20:03:32,687 INFO: PyABSA version: 2.4.2


INFO:bert_mlp:PyABSA version: 2.4.2


2026-05-17 20:03:32,689 INFO: Transformers version: 5.0.0


...

INFO:bert_mlp:Device: Unknown


[2026-05-17 20:03:32] (2.4.2)

/usr/lib/python3.12/multiprocessing/pool.py:268: ResourceWarning: unclosed running multiprocessing pool <multiprocessing.pool.Pool state=RUN pool_size=1>
  _warn(f"unclosed running multiprocessing pool {self!r}",


 Try to load ['datasets/sentiment'] dataset from local disk


FileNotFoundError: [Errno 2] No such file or directory: ''

In [11]:
import os

print(os.listdir('datasets/sentiment'))
print("\nConteúdo do train.txt (primeiras 3 linhas):")
with open('datasets/sentiment/train.txt', 'r') as f:
    for i, line in enumerate(f):
        if i < 3:
            print(line.strip())


['test.txt', 'train.txt']

Conteúdo do train.txt (primeiras 3 linhas):
sou muito grata por vcs serem tão competentes parabéns obgada$1
bom dia ainda não recebi toa a minha encomenda pos fiz o pedido de relógios de parede ate agora so chgaram$-1
muito bonito chegou no prazo poderiam ter indicado o número do pedido na embalagem para saber$1


In [12]:
from pyabsa.tasks.TextClassification import TCTrainer, TCConfigManager, BERTTCModelList
from pyabsa import ModelSaveOption

config = TCConfigManager.get_tc_config_english()
config.model = BERTTCModelList.BERT
config.pretrained_bert = 'neuralmind/bert-base-portuguese-cased'
config.num_epoch = 3
config.batch_size = 16
config.max_seq_len = 128
config.seed = 42
config.log_step = 100
config.evaluate_begin = 0
config.save_mode = ModelSaveOption.SAVE_MODEL_STATE_DICT

trainer = TCTrainer(
    config=config,
    dataset='/content/datasets/sentiment',
    checkpoint_save_mode=ModelSaveOption.SAVE_MODEL_STATE_DICT,
    auto_device=True
)

[2026-05-17 20:07:00] (2.4.2) Set Model Device: cpu
[2026-05-17 20:07:00] (2.4.2) Device Name: Unknown


2026-05-17 20:07:00,604 INFO: PyABSA version: 2.4.2


INFO:bert_mlp:PyABSA version: 2.4.2


2026-05-17 20:07:00,607 INFO: Transformers version: 5.0.0


INFO:bert_mlp:Transformers version: 5.0.0


...

INFO:bert_mlp:Torch version: 2.10.0+cpu+cudaNone


2026-05-17 20:07:00,609 INFO: Device: Unknown


INFO:bert_mlp:Device: Unknown


[2026-05-17 20:07:00] (2.4.2) Try to load ['/content/datasets/sentiment'] dataset from local disk


FileNotFoundError: [Errno 2] No such file or directory: ''

In [13]:
import pyabsa
print(pyabsa.__version__)

# Verificar o formato esperado
from pyabsa.utils.data_utils.dataset_manager import detect_dataset
help(detect_dataset)

2.4.2
Help on function detect_dataset in module pyabsa.utils.data_utils.dataset_manager:

detect_dataset(dataset_name_or_path, task_code: pyabsa.framework.flag_class.flag_template.TaskCodeOption = None, load_aug=False, config=None, **kwargs)
    Detect dataset from dataset_path, you need to specify the task type, which can be TaskCodeOption.Aspect_Polarity_Classification, 'atepc' or 'tc', etc.

    :param dataset_name_or_path: str or DatasetItem
        The name or path of the dataset.
    :param task_code: str or TaskCodeOption
        The task type, such as "apc" for aspect-polarity classification or "tc" for text classification.
    :param load_aug: bool, default False
        Whether to load the augmented dataset.
    :param config: Config, optional
        The configuration object.
    :param kwargs: dict
        Additional keyword arguments.

    :return: dict
        A dictionary containing file paths for the train, test, and validation sets.



In [14]:
from pyabsa.tasks.TextClassification import TCTrainer, TCConfigManager, BERTTCModelList
from pyabsa import ModelSaveOption, TaskCodeOption

config = TCConfigManager.get_tc_config_english()
config.model = BERTTCModelList.BERT
config.pretrained_bert = 'neuralmind/bert-base-portuguese-cased'
config.num_epoch = 3
config.batch_size = 16
config.max_seq_len = 128
config.seed = 42
config.log_step = 100
config.evaluate_begin = 0
config.save_mode = ModelSaveOption.SAVE_MODEL_STATE_DICT

trainer = TCTrainer(
    config=config,
    dataset='/content/datasets/sentiment',
    checkpoint_save_mode=ModelSaveOption.SAVE_MODEL_STATE_DICT,
    auto_device=True,
    task_code=TaskCodeOption.Text_Classification
)

TypeError: TCTrainer.__init__() got an unexpected keyword argument 'task_code'

In [15]:
from pyabsa.tasks.TextClassification import TCTrainer, TCConfigManager, BERTTCModelList
from pyabsa import ModelSaveOption

config = TCConfigManager.get_tc_config_english()
config.model = BERTTCModelList.BERT
config.pretrained_bert = 'neuralmind/bert-base-portuguese-cased'
config.num_epoch = 3
config.batch_size = 16
config.max_seq_len = 128
config.seed = 42
config.log_step = 100
config.evaluate_begin = 0
config.save_mode = ModelSaveOption.SAVE_MODEL_STATE_DICT

dataset = {
    'train': '/content/datasets/sentiment/train.txt',
    'test': '/content/datasets/sentiment/test.txt'
}

trainer = TCTrainer(
    config=config,
    dataset=dataset,
    checkpoint_save_mode=ModelSaveOption.SAVE_MODEL_STATE_DICT,
    auto_device=True
)

[2026-05-17 20:10:02] (2.4.2) Set Model Device: cpu
[2026-05-17 20:10:02] (2.4.2) Device Name: Unknown


AttributeError: 'TCConfigManager' object has no attribute 'dataset_name'

In [16]:
!pip install transformers datasets scikit-learn -q

In [17]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

# Carregar tokenizer do BERTimbau
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

print("✅ Tokenizer carregado!")

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/bert/tokenization_bert.py:104: DeprecationWarning: Deprecated in 0.9.0: WordPiece.__init__ will not create from files anymore, try `WordPiece.from_file` instead
  self._tokenizer = Tokenizer(WordPiece(self._vocab, unk_token=str(unk_token)))


✅ Tokenizer carregado!


In [18]:
class SentimentDataset(Dataset):
    def __init__(self, textos, rotulos, tokenizer, max_len=128):
        self.textos = textos
        self.rotulos = rotulos
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.textos[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.rotulos[idx], dtype=torch.long)
        }

# Mapear sentimento para índice 0, 1, 2
def mapear_label(s):
    if s == 'negativo':
        return 0
    elif s == 'neutro':
        return 1
    else:
        return 2

train_df['label'] = train_df['sentimento'].apply(mapear_label)
test_df['label'] = test_df['sentimento'].apply(mapear_label)

train_dataset = SentimentDataset(
    train_df['texto'].tolist(),
    train_df['label'].tolist(),
    tokenizer
)

test_dataset = SentimentDataset(
    test_df['texto'].tolist(),
    test_df['label'].tolist(),
    tokenizer
)

print("✅ Datasets criados!")
print("Treino:", len(train_dataset))
print("Teste:", len(test_dataset))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✅ Datasets criados!
Treino: 28196
Teste: 7050


In [19]:
from transformers import BertForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Carregar modelo
model = BertForSequenceClassification.from_pretrained(
    'neuralmind/bert-base-portuguese-cased',
    num_labels=3
)

# Função de métricas
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# Configurar treinamento
training_args = TrainingArguments(
    output_dir='./modelo_sentimento',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=100,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("✅ Modelo e trainer configurados!")

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7bdccda4fd90>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7bdccda4c360>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7bdccda4d4e0>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7bdccda4d080>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7bdccda4db00>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7bdccda4cd70>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7bdccda4c830>


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7bdcf7c269e0>
BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MI

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [20]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Função de métricas
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

# Configurar treinamento
training_args = TrainingArguments(
    output_dir='./modelo_sentimento',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=100,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("✅ Trainer configurado!")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer configurado!


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [1]:
!pip install transformers -q

import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Carregar dados
df = pd.read_csv('olist_processado.csv')

# Mapear labels
def mapear_label(s):
    if s == 'negativo':
        return 0
    elif s == 'neutro':
        return 1
    else:
        return 2

df['label'] = df['sentimento'].apply(mapear_label)

# Dividir treino e teste
train_df, test_df = train_test_split(
    df[['texto', 'label']],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Dataset PyTorch
class SentimentDataset(Dataset):
    def __init__(self, textos, rotulos, tokenizer, max_len=128):
        self.textos = textos
        self.rotulos = rotulos
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.textos[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.rotulos[idx], dtype=torch.long)
        }

# Tokenizer e modelo
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
model = BertForSequenceClassification.from_pretrained(
    'neuralmind/bert-base-portuguese-cased',
    num_labels=3
)

train_dataset = SentimentDataset(train_df['texto'].tolist(), train_df['label'].tolist(), tokenizer)
test_dataset = SentimentDataset(test_df['texto'].tolist(), test_df['label'].tolist(), tokenizer)

print("✅ Tudo pronto!")
print("Treino:", len(train_dataset), "| Teste:", len(test_dataset))
print("GPU disponível:", torch.cuda.is_available())

FileNotFoundError: [Errno 2] No such file or directory: 'olist_processado.csv'

In [2]:
!pip install transformers -q

import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Carregar dados
df = pd.read_csv('olist_processado.csv')

# Mapear labels
def mapear_label(s):
    if s == 'negativo':
        return 0
    elif s == 'neutro':
        return 1
    else:
        return 2

df['label'] = df['sentimento'].apply(mapear_label)

# Dividir treino e teste
train_df, test_df = train_test_split(
    df[['texto', 'label']],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Dataset PyTorch
class SentimentDataset(Dataset):
    def __init__(self, textos, rotulos, tokenizer, max_len=128):
        self.textos = textos
        self.rotulos = rotulos
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.textos[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.rotulos[idx], dtype=torch.long)
        }

# Tokenizer e modelo
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
model = BertForSequenceClassification.from_pretrained(
    'neuralmind/bert-base-portuguese-cased',
    num_labels=3
)

train_dataset = SentimentDataset(train_df['texto'].tolist(), train_df['label'].tolist(), tokenizer)
test_dataset = SentimentDataset(test_df['texto'].tolist(), test_df['label'].tolist(), tokenizer)

print("✅ Tudo pronto!")
print("Treino:", len(train_dataset), "| Teste:", len(test_dataset))
print("GPU disponível:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

✅ Tudo pronto!
Treino: 28196 | Teste: 7050
GPU disponível: True


In [3]:
# Configurar treinamento
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./modelo_sentimento',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=100,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.434240,0.389180,0.854326,0.819284
2,0.367441,0.397105,0.860284,0.847696
3,0.258222,0.457511,0.855035,0.850590


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5289, training_loss=0.3475909154657201, metrics={'train_runtime': 2119.229, 'train_samples_per_second': 39.915, 'train_steps_per_second': 2.496, 'total_flos': 5564059444694016.0, 'train_loss': 0.3475909154657201, 'epoch': 3.0})

In [4]:
# Salvar modelo e tokenizer
model.save_pretrained('./modelo_sentimento_final')
tokenizer.save_pretrained('./modelo_sentimento_final')

print("✅ Modelo salvo em ./modelo_sentimento_final")
print(os.listdir('./modelo_sentimento_final'))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modelo salvo em ./modelo_sentimento_final


NameError: name 'os' is not defined

In [5]:
import os
print("✅ Arquivos do modelo:")
print(os.listdir('./modelo_sentimento_final'))

✅ Arquivos do modelo:
['model.safetensors', 'config.json', 'tokenizer_config.json', 'tokenizer.json']


In [6]:
import shutil

# Compactar pasta do modelo
shutil.make_archive('modelo_sentimento_final', 'zip', './modelo_sentimento_final')

print("✅ Arquivo compactado: modelo_sentimento_final.zip")

✅ Arquivo compactado: modelo_sentimento_final.zip


In [7]:
from google.colab import files

files.download('modelo_sentimento_final.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!pip install transformers torch -q

from transformers import BertForSequenceClassification, BertTokenizer
import torch

# Carregar modelo e tokenizer
model = BertForSequenceClassification.from_pretrained('Amand4priscil4/promosense-modelo')
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

model.eval()

# Função de predição
def prever_sentimento(texto):
    inputs = tokenizer(
        texto,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()
    labels = {0: '😡 negativo', 1: '😐 neutro', 2: '😊 positivo'}
    return labels[pred]

# Testes
textos = [
    "produto chegou antes do prazo, adorei!",
    "péssimo produto, veio quebrado e o vendedor não respondeu",
    "produto ok, nada demais",
    "ótimo custo benefício, recomendo muito!",
    "demorou demais para entregar, decepcionante"
]

for texto in textos:
    print(f"Texto: {texto}")
    print(f"Sentimento: {prever_sentimento(texto)}")
    print()


OSError: Amand4priscil4/promosense-modelo does not appear to have a file named pytorch_model.bin or model.safetensors.

In [9]:
from huggingface_hub import HfApi
from google.colab import files

# Fazer login
from huggingface_hub import login
login()


In [10]:
from huggingface_hub import HfApi

api = HfApi()

# Fazer upload dos arquivos do modelo
api.upload_folder(
    folder_path='./modelo_sentimento_final',
    repo_id='Amand4priscil4/promosense-modelo',
    repo_type='model'
)

print("✅ Modelo enviado com sucesso!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...o_final/model.safetensors:   0%|          |  173kB /  436MB            

✅ Modelo enviado com sucesso!


In [11]:
from transformers import BertForSequenceClassification, BertTokenizer
import torch

# Carregar modelo e tokenizer
model = BertForSequenceClassification.from_pretrained('Amand4priscil4/promosense-modelo')
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

model.eval()

# Função de predição
def prever_sentimento(texto):
    inputs = tokenizer(
        texto,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()
    labels = {0: '😡 negativo', 1: '😐 neutro', 2: '😊 positivo'}
    return labels[pred]

# Testes
textos = [
    "produto chegou antes do prazo, adorei!",
    "péssimo produto, veio quebrado e o vendedor não respondeu",
    "produto ok, nada demais",
    "ótimo custo benefício, recomendo muito!",
    "demorou demais para entregar, decepcionante"
]

for texto in textos:
    print(f"Texto: {texto}")
    print(f"Sentimento: {prever_sentimento(texto)}")
    print()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Texto: produto chegou antes do prazo, adorei!
Sentimento: 😊 positivo

Texto: péssimo produto, veio quebrado e o vendedor não respondeu
Sentimento: 😡 negativo

Texto: produto ok, nada demais
Sentimento: 😊 positivo

Texto: ótimo custo benefício, recomendo muito!
Sentimento: 😊 positivo

Texto: demorou demais para entregar, decepcionante
Sentimento: 😡 negativo

